# Data Driven Analytics Summative 2: Technical Implementation

## Data Acquisition and Engineering

In [22]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "ulrikthygepedersen/online-retail-dataset",
    "online_retail.csv",
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [23]:
df.shape

(541909, 8)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### Missing Values

In [25]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

#### Missing Stock Codes

In [26]:
df.shape

(541909, 8)

In [27]:
# Count unique descriptions per StockCode
desc_counts = df.groupby("StockCode")["Description"].nunique()

# StockCodes with more than one description
multi_desc_stockcodes = desc_counts[desc_counts > 1]

# Inspect
multi_desc_stockcodes.sort_values(ascending=False).head(10)


StockCode
20713     8
23084     7
85175     6
21830     6
85172     5
21181     5
72807A    5
23131     5
23343     5
85123A    4
Name: Description, dtype: int64

In [28]:
# StockCodes with missing descriptions
missing_desc_stockcodes = df.loc[
    df["Description"].isnull(), "StockCode"
].unique()

ambiguous_stockcodes = multi_desc_stockcodes.index.intersection(
    missing_desc_stockcodes
)

print(f"Number of ambiguous StockCodes: {len(ambiguous_stockcodes)}")

Number of ambiguous StockCodes: 174


In [29]:
df[df["StockCode"].isin(ambiguous_stockcodes)][
    ["StockCode", "Description"]
].drop_duplicates().sort_values("StockCode")

,StockCode,Description
103332,10080,GROOVY CACTUS INFLATABLE
454350,10080,check
279310,10080,NaN
23437,15058A,BLUE POLKADOT GARDEN PARASOL
192289,15058A,wet/rusty
...,...,...
10792,90014B,GOLD M PEARL ORBIT NECKLACE
386649,90014B,GOLD M.O.P. ORBIT NECKLACE
142063,90014C,SILVER/BLACK ORBIT NECKLACE
381682,90014C,NaN


Stock Code is complete, description is not. Null values will be filled with the appropriate description based on stock code. In stock codes where more than one description is present, the most common description is used.

In [30]:
# Create a mapping from StockCode to the most common non-null Description
stockcode_description_map = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .agg(lambda x: x.mode().iloc[0])
)

df["Description"] = df["Description"].fillna(
    df["StockCode"].map(stockcode_description_map)
)

df["Description"].isnull().sum()

112

Records where no historical description existed for the StockCode were retained but flagged to preserve transaction integrity

In [31]:
df["Description"] = df["Description"].fillna("UNKNOWN_PRODUCT")
df["Description"].isnull().sum()

0

In [32]:
df.shape

(541909, 8)

An exploratory consistency check revealed that a small subset of StockCodes mapped to multiple product descriptions, indicating historical inconsistency in the source data.
For StockCodes with missing descriptions, the most frequently occurring description was used to minimise semantic ambiguity.
StockCodes with no historical descriptions were retained and flagged as unknown to preserve transactional integrity.

#### Missing CustomerID

There are 135080 missing customerID rows. It is not feasible to simply drop these rows without losing large amounts of information, inputting random values would destroy the churn analysis and using mean values would not make sense. Because CustomerID is assigned at the invoicelevel and each invoice can span multiple rows, we can attempt to recover missing CustomerIDs by looking at other rows with the same InvoiceNo.

In [33]:
# Build a mapping: InvoiceNo → CustomerID
invoice_customer_map = (
    df.dropna(subset=["CustomerID"])
      .groupby("InvoiceNo")["CustomerID"]
      .agg(lambda x: x.mode().iloc[0])
)

df["CustomerID"] = df["CustomerID"].fillna(
    df["InvoiceNo"].map(invoice_customer_map)
)

df["CustomerID"].isnull().sum()

135080

You attempted recovery using key‑based logic
You validated whether recovery was possible
You concluded (correctly) that it was not feasible without inventing data. 

To preserve the data, these rows will be kept. It was considered to fill these rows with "Guest User" however this would impact the datatype of the column and could cause issues later in the pipeline. I also considered filling with -1, this would preserve the datatype, but could cause issues in any aggregation.

As such, it is best to leave these values as null and create a binary customer type flag where 1 represents a registered customer and 0 represents a guest user. This allows the full dataset to be used for transaction modelling, but excludes these null values from customer churn modelling, preserving quality.

Using 0 and 1 (and creating this flag variable), is more suitable as this is a modelling workbook. This removes the need to encode this column later.

In [34]:
#df["CustomerID"] = df["CustomerID"].fillna("Guest User")
#df["CustomerID"].isnull().sum()

In [35]:
import numpy as np

df["CustomerTypeFlag"] = np.where(
    df["CustomerID"].isna(),
    0,
    1
)

In [36]:
df["CustomerTypeFlag"].value_counts()

CustomerTypeFlag
1    406829
0    135080
Name: count, dtype: int64

In [37]:
df.shape

(541909, 9)

### Duplicated Values

In [38]:
df.duplicated(subset=None).sum()

5268

In [39]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,CustomerTypeFlag
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,1
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,1
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,1
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,1
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,1
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,1


In [40]:
df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,CustomerTypeFlag
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom,1
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,1
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom,1
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom,1
...,...,...,...,...,...,...,...,...,...
440149,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,2011-11-04 13:25:00,2.95,15110.0,United Kingdom,1
461407,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838.0,United Kingdom,1
461408,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838.0,United Kingdom,1
529980,C580764,22667,RECIPE BOX RETROSPOT,-12,2011-12-06 10:38:00,2.95,14562.0,United Kingdom,1


In [41]:
df.drop_duplicates(inplace=True)

In [42]:
df.shape

(536641, 9)

Duplicate rows were examined and as they were shown to be exact matches were removed. As a customer can have a quantity of more than 1, it suggests that these rows are errors rather than legitimate orders, as such they have been dropped from the dataset.